### Neshyba 2026


# Conjugated polyenes (and their $\pi$-orbital electrons)

## Introduction

### Solution to a conjugated polyene problem using modern electronic structure software

Some types of molecular orbitals exhibit extensive $\pi$-system *delocalization*. As you might know already, this occurs whenever there's an alternating pattern of single and double bonds, such as occurs in the molecule $C_{10}H_{12}$.

<p style='text-align: center;'>
<img src="http://webspace.pugetsound.edu/facultypages/nesh/Notebook/delocalized pi-orbital of C10H12.jpg" height="700" width="700"/>
<strong>Figure 1</strong>. $\psi_{HOMO}$ of the conjugated polyene $C_{10}H_{12}$, according to Spartan<sup>TM</sup>. 
</p>

The information presented in Fig. 1 was obtained using the electronic structure software Spartan<sup>TM</sup>, which solves Schrödinger's time-independent equation 

$$
H\psi=E\psi \ \ \ \ (1)
$$

where $H=-{\hbar^2 \over 2m} \nabla^2+V$, the Hamiltonian operator for the problem at hand. The particular orbital shown in Fig. 1 is labeled $\psi_{HOMO}$ because it's the *Highest Occupied Molecular Orbital*. (Aside: the orbital whose energy lies just above $\psi_{HOMO}$ would be called $\psi_{LUMO}$, for *Lowest Unoccupied Molecular Orbital*.)

### Inferring the de Broglie wavelength from delocalized $\pi$-system orbitals

A cool thing about depictions such as Fig. 1 is that you can use them to infer the de Broglie wavelength of the electrons that occupy such orbitals. How? For $\pi$-type orbitals like this one, $\lambda_{de Broglie}$ can be approximated as the distance between lobes of the same phase (color) as you proceed from one end of the molecule to another. In the above example, this is about half the length of the molecule. In fact, to complete this CGI, you'll need to solve the electronic structure of $C_{10}H_{12}$ in Spartan<sup>TM</sup>, locate $\psi_{HOMO}$, and make some distance measurements. You'll also need an estimate of the distance (call it "$a$") of the entire length of the molecule, since it appears that $\pi$-system electrons are delocalized over that entire distance.

Why do we care about the de Broglie wavelength? Knowing its value gives us a big clue about the speed and kinetic energy of an electron. This is evident in the de Broglie equation, 

$$
\lambda_{de Broglie} = {h \over m v} \ \ \ \ (2a)
$$

or equivalently

$$
v = {h \over m \lambda_{de Broglie}} \ \ \ \ (2b)
$$

where $h$ is Planck's constant (of course), $m$ is the mass of the electron, and $v$ is its speed. It's clear from the de Broglie relation that smaller $\lambda_{de Broglie}$ implies a faster-moving electron. Furthermore, once we know an electron's speed, we can estimate its kinetic energy, 

$$
KE_{de Broglie} = {1 \over 2} m v^2 \ \ \ \ (3)
$$

Equation 3 will make it possible to compare de Broglie's predictions about the energy of electrons, to results obtained via other means.

### An exact solution to a simplified model: the Particle-on-a-line

Spartan<sup>TM</sup>'s solution of Schrödinger's equation doesn't take very long on modern computers, but this capability (and the software itself) only became widely available with the advent of personal computers. Before then, people had to rely on simpler models that they could solve "by hand," which (they hoped) would still give  insight into phenomena they were interested in. In the *Particle-on-a-line* model, for example, one pretends that an electron has unimpeded access to the entire molecule, not even noticing when it passes by the carbon atoms! Mathematically, this corresponds to a potential energy of zero across the length of the molecule, rising to infinity at the ends (so that the electron can't hop off the molecule!). 

Nobody thinks that the Particle-on-a-line model is an exact representation of delocalized $\pi$ electrons on a conjugated polyene like $C_{10}H_{12}$. However, it turns out that this problem can be solved exactly, with pen an paper! The result is a set of *eigenenergies*,

$$
E_n = {n^2 h^2 \over 8ma^2} \ \ \ \ (4)
$$

where $m$ is the mass of the electron and $n=1, 2, ....$  

### Linear Algebra as a way of solving Schrödinger's equation

So how *does* Spartan solve Schrödinger's equation? Without getting into great detail, the answer is that the methods of Linear Algebra are used. More specifically, the $H$ in Eq. 1 is represented as a matrix, and that matrix is then "diagonalized". Diagonalizing gives us wavefunctions -- the $\psi$ functions appearing in Eq. 1. Diagonalizing also gives us the eigenenergies of the model -- the $E$ values appearing in Eq. 1. And Python has a diagonalizing package! It's called scipy.linalg.eigh. So that means we can do some of what Spartan does right here in Python. And it also allows us to get answers to model systems that we can't do with pen and paper, which Spartan isn't set up to do. So it's like a little sandbox that we can play in.

Downsides to diagonalizing? Well, the big one is that diagonalizing is an approximate method. In the present situation, its accuracy depends on how finely we choose to make the grid of x-values we're using to represent the line. In the code below, this is defined by the variable *nsteps*; in practice, as long as we keep *nstep* above 200 points or so, we can get satisfactory results. The other downside (which is really too bad) is that the eigenvalues and wave functions provided by diagonalization are provided numerically. That means we can never expect a formula like Eq. 4 to emerge from Spartan or scipy.linalg.eigh.

### The idea of this exercise
To summarize, what we're going to do here is to treat the Spartan result as "the truth," as if it were an experimental result, and then pose questions like how how good is the Particle-on-a-line model at simulating that system. We'll also develop the idea of de Broglie's formula (Eqs. 2a and 2b), just because it helps build up an intuition about what wavefunctions are really telling us. Hopefully, the result of all this thinking will add up to a better intuition about the quantum mechanics of delocalized electrons, and the methodologies used to make predictions.

### Learning goals
The main learning goals of this exercise are 
1. I can describe key assumptions associated with the *Particle-on-a-line* model.
1. I can relate solutions of the *Particle-on-a-line* model to delocalized $\pi$-electrons orbitals of conjugated polyenes.
1. I know what *HOMO* and *LUMO* mean.
1. I can describe what happens, qualitatively, to *Particle-on-a-line* energies as the reference potential is raised or lowered, and as the line gets longer or shorter.

In [ ]:
import pint; from pint import UnitRegistry; AssignQuantity = UnitRegistry(system='atomic').Quantity
import numpy as np
import scipy.linalg as spla
import matplotlib.pyplot as plt
import pchemlibrary as PL
import plotly.graph_objects as go
%matplotlib inline

In [ ]:
# Quantum constants
hbar = AssignQuantity(1,'atomic_unit_of_time * hartree'); print(hbar)
h = hbar*2*np.pi; print(h)
m = AssignQuantity(1,'atomic_unit_of_mass'); print(m)

### Getting the kinetic energy from the de Broglie wavelength
You'll need the de Broglie wavelength of $\psi_{HOMO}$, as inferred from Spartan, for the cell below.

In [ ]:
# Specify the de Broglie wavelength of psi_HOMO of C10H12 (from Spartan); call your result "lamb_dB"
# Your code here


# Use the de Broglie formula to get a speed (using Eq. 2b)
# Your code here


# Calculate and print this electron's kinetic energy (Eq. 3)
# Your code here


### Comparison of Particle-on-a-line electron energy with de Broglie 
In the cell below, our goal is to get the energies of states of the Particle-on-a-line model. To do that, you'll need to return to Spartan and measure the length of our $C_{10}H_{12}$ molecule, by adding up all the carbon-carbon distances from end to end. Because Spartan gives you these distances in Angstroms, it's convenient to do the adding-up in Angstroms, then use Pint to convert the sum to bohrs.

The second thing we'll do here is to use the particle-on-a-line result (Eq. 4) to calculate the energy of the HOMO. Which wavefunction is the HOMO? Hint: If you look at the structure $C_{10}H_{12}$, you'll see there are five $\pi$ bonds, each containing two electrons, and each wavefunction can accommodate two electrons.

In [ ]:
# Specify the length of the molecule ("a") using your Spartan result.
# Your code here


# Calculate and print the energy of the HOMO ("E_HOMO") using our analytical result (Eq. 4)
# Your code here


### Pause for analysis
Approximately how close (in %) did the energy of the HOMO as produced by the particle-in-a-line result compare to the de Broglie estimate?

YOUR ANSWER HERE

### Comparison of Particle-on-a-line HOMO-LUMO gap with Spartan's HOMO-LUMO gap
Next we'll investigate the HOMO-LUMO gap. 

In [ ]:
# For the gap, we need the HOM and LUMO energies from the Particle-on-a-line
E_HOMO = 5**2*h**2/(8*m*a**2)
E_HOMO.ito('hartree')
E_LUMO = 6**2*h**2/(8*m*a**2)
E_LUMO.ito('hartree')

# Now the gap according to Particle-on-a-line theory
E_gap = E_LUMO-E_HOMO
print('Particle-on-a-line result for the HOMO-LUMO gap: ', E_gap)

# Now let's see what Spartan says about this (convert to hartrees, and print).
# Your code here


### Pause for analysis
Approximately how close (in %) did the energy of the HOMO as produced by the particle-in-a-line result compare to the Spartan result?

YOUR ANSWER HERE

### The "shortcut" correction
Assuming you got a HOMO-LUMO gap for the Particle-on-a-line that was smaller than that of Spartan (which, remember, we're considering the "truth" here), how shall we correct for it? One thought is, the wave functions tend to short-cut the zigs and zags of carbon-carbon bonds, so the effective length of the molecule is shorter than what we'd estimate by adding up all the carbon-carbon bond lengths. But how much shorter? 

The cell below does this comparison by applying a factor ("shortcutfactor") to the length of the molecule. Specify some factor (between one and zero, we presume) and execute the cell to see if you get a good match to the Spartan result.

In [ ]:
# Specify a value of shortcutfactor
# Your code here


# Apply the shortcut to the box length
ashorter = a * shortcutfactor

# Get the new HOMO and LUMO energies from the Particle-on-a-line
E_HOMO = 5**2*h**2/(8*m*ashorter**2)
E_HOMO.ito('hartree')
E_LUMO = 6**2*h**2/(8*m*ashorter**2)
E_LUMO.ito('hartree')

# Now the gap according to Particle-on-a-line theory
E_gap = E_LUMO-E_HOMO
print('Particle-on-a-line result for the HOMO-LUMO gap: ', E_gap)

# Now let's see what Spartan says about this (convert to hartrees, and print).
E_HOMO_Spartan = AssignQuantity(-5.0,'eV')
E_LUMO_Spartan = AssignQuantity(-1.9,'eV')
E_gap_Spartan = E_LUMO_Spartan-E_HOMO_Spartan
E_gap_Spartan.ito('hartree')
print('Spartan result for the HOMO-LUMO gap: ', E_gap_Spartan)
error = (E_gap-E_gap_Spartan)/E_gap_Spartan*100
print('Error (%) with respect to Spartan =', error) 

### Numerical work: Laying out an array of x-values
The cell below lays out an array of x-values, and the difference between x-values ($dx$). You eventually might want to change the number of steps to get better numerial accuracy, but for now this is good.

In [ ]:
# Create a grid of points across the line
nsteps = 200
xvec=np.linspace(0,a,nsteps)
dx = xvec[1]-xvec[0]

### Numerical work: solving the particle-on-a-line model  by diagonalizing the Hamiltonian matrix
The cell below calculates the energies and wave functions for the particle-on-a-line model by constructing a matrix representation the Hamiltonian operator, and solving Schrödinger's equation using linear algebra. It's not expected that you understand all the details here, but because we'll re-use some of these ideas, it's good to be familiar with the sequence.

In [ ]:
# Create the kinetic part of the Hamiltonian
KE=-0.5*hbar**2/m*(-2.0*np.diag(np.ones(nsteps))+np.diag(np.ones(nsteps-1),1)+np.diag(np.ones(nsteps-1),-1))/dx**2

# Create a potential energy matrix, whose values are all zero
PE = PL.flat_potential(xvec, V0=0)
PE = AssignQuantity(PE,'hartree')

# Create the Hamiltonian
H = KE + PE
H.ito('hartree')

# Diagonalize the Hamiltonian yielding the wavefunctions and energies
Epsi,psi = spla.eigh(H)
Epsi = AssignQuantity(Epsi,'hartree')

# Specify the number of wavefunctions we want to look at
number_of_wavefunctions = 7

# Plot the first few wavefunctions
PL.plotter(Epsi,psi,xvec,PE,number_of_wavefunctions)

### Pause for analysis

1. Have a close look at these wavefunctions, and come up with a formula that relates the *number of peaks* to the value of $n$. 
1. Go back into Spartan and compare $\psi_{HOMO}$ and $\psi_{LUMO}$ to the figure you generated in the cell above. What differences or similarities do you see?
1. Given that the numerically-obtained energies you just got, and the analytically-obtained energies you got from Eq. 4, describe the same exact model (the particle-in-a-line), they should give *exactly* the same results, right? If you look carefully at these energies, however, you'll see some small differences. What do you suppose is the origin of these differences, and how would you go about testing that supposition, and how would you go about testing that supposition?
1. Yale physical chemist William Chupka often claimed that, rather than avoiding nodes, instead "electrons go through nodes." What do you do you suppose he meant by that in the present context?

YOUR ANSWER HERE

### Visualizing the Hamiltonian matrix
As you probably noticed, the matrix representation of the Hamiltonian has a potential energy part, which lies on the diagonal, and a kinetic energy part, which has diagonal and off-diagonal elements. There's a nice description of the theory behind this at https://github.com/DalInar/schrodingers-snake/blob/master/Finite%20Difference%20Diagonalization/Finite%20Difference%20Diagonalization%20-%20Solution.ipynb. 

It's also useful to visualize the Hamiltonian matrix. We do this in the cell below. The structure you see is all due to the Laplacian, since $V=0$ for this model. We're plotting just the first few elements so you can see the pieces more easily; if you increase $N$, you can see more of the Hamiltonian, but the structure will be less resolved.

In [ ]:
N = 10
fig = go.Figure(data=go.Surface(x=xvec[0:N],y=xvec[0:N],z=H[0:N,0:N]))
fig.update_layout(scene = dict(
                    xaxis_title="x",
                    yaxis_title="x'",
                    zaxis_title='H'))

### About the reference value of the potential energy
As you might remember from a previous course, in classical mechanics, the trajectory of a moving object is not affected by the absoute, or reference, value of the potential energy -- as in, playing ultimate frizbee on a rootop would be just the same as on Todd field, right? In quantum mechanics, we don't have trajectories, but we do have wavefunctions and eigenenergies. How does changing the absoute value of the potential energy affect those quantities? 

To answer this question, duplicate what you just did, but change the reference potential ("V0" in the call to the flat potential energy function) to some other value -- say 100 hartree instead of zero.

In [ ]:
# Create the kinetic part of the Hamiltonian
# Your code here


# Create a potential energy part (all values are zero, but still)
# Your code here


# Create the Hamiltonian
# Your code here


# Diagonalize the Hamiltonian yielding the wavefunctions and energies
# Your code here


# Specify the number of wavefunctions we want to look at
# Your code here


# Plot the first few wavefunctions
# Your code here


### Pause for analysis
So ... what changes in quantum mechanics, and what doesn't change, when the reference potential is changed?   

YOUR ANSWER HERE

### Visualizing the Hamiltonian matrix with a non-zero (but still constant) potential energy
The cell below is for visualization purposes only -- to see what the Hamiltonian matrix looks like when there's a different (but still constant) potential energy. The structure you see is all due to the Laplacian, since $V=0$ for this model.

In [ ]:
N = 10
fig = go.Figure(data=go.Surface(x=xvec[0:N],y=xvec[0:N],z=H[0:N,0:N]))
fig.update_layout(scene = dict(
                    xaxis_title="x",
                    yaxis_title="x'",
                    zaxis_title='H'))

### Pause for analysis
So ... how is the Hamiltonian matrix different when the reference potential is changed? You should be aware that plotly autoscales the vertical axis to accommodate the full range of the function being plotted.

YOUR ANSWER HERE

### Refreshing and saving your code
1. Use the dropdown menu Kernel/Restart
2. Use the dropdown menu Cell/Run All Above
3. Under the "File" dropdown menu item in the upper left is a disk icon. Press it now to save your work (you can, do this at any time as you're working on an assignment, actually).

### Validating
This step will help ensure that you didn't miss something (although it's not a guarantee). Find the "Validate" button and press it. If there are any errors or warnings, fix them.

### Finishing up
Assuming all this has gone smoothly, carry out three more steps (but read this carefully before starting):
1. Close this notebook using the "File/Close and Halt" dropdown menu
1. Using the Assignments tab, submit this notebook
1. Press the Logout tab of the Home Page